# Solución de ecuaciones no lineales por métodos numéricos

## Caso de estudio: el punto de retorno seguro — Montería Drone Delivery

**Autores:** Santiago Anaya Ramos y Andrés David Cuello Acosta
**Asignatura:** Métodos Numéricos
**Métodos analizados:** Bisección · Regla Falsa · Newton-Raphson

---

### Resumen

Este informe determina la raíz de la ecuación no lineal

$$
f(x) = D\ln(x) - \frac{E}{10}\,e^{0.02x}
$$

mediante tres métodos numéricos, y compara su comportamiento en número de
iteraciones, precisión alcanzada y velocidad de convergencia.

La función se define **simbólicamente** con SymPy, de modo que la derivada
necesaria para Newton-Raphson se obtiene de forma automática y no puede diferir
de la función usada por los algoritmos. Cada iteración se documenta con su
desarrollo matemático completo.

Los parámetros $D$ y $E$ son configurables: el informe se regenera por completo
al cambiarlos en la celda de parámetros (sección 2.1) y volver a ejecutar el
documento.

### Contenido

1. Modelo matemático
2. Configuración del caso analizado
3. Bisección
4. Regla Falsa
5. Newton-Raphson
6. Comparación de los tres métodos
7. Convergencia
8. Conclusiones
9. Anexo — código de los algoritmos

In [ ]:
%pip -q install ipywidgets sympy

In [ ]:
import inspect
import re
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Math, Markdown

pd.set_option("display.precision", 6)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# 1. Modelo matemático

La función que modela el balance de energía del dron es

$$
f(x) = D\ln(x) - \frac{E}{10}\,e^{0.02x}, \qquad x > 0
$$

donde $x$ es el tiempo en segundos y el dominio se restringe a $x > 0$ por la
presencia de $\ln(x)$. La raíz $r$ tal que $f(r) = 0$ corresponde al **punto de
retorno seguro**.

Newton-Raphson requiere además la derivada, que se obtiene por derivación
simbólica de la expresión anterior:

$$
f'(x) = \frac{\partial}{\partial x}\left(D\ln(x) - \frac{E}{10}e^{0.02x}\right)
      = \frac{D}{x} - 0.002\,E\,e^{0.02x}
$$

In [ ]:
# Definición simbólica: única fuente de verdad de todo el documento.
x = sp.symbols("x", positive=True)      # positive=True por la presencia de ln(x)
D_sym, E_sym = sp.symbols("D E")        # parámetros del modelo

f_expr = D_sym * sp.log(x) - (E_sym / 10) * sp.exp(sp.Float("0.02") * x)
df_expr = sp.diff(f_expr, x)            # la derivada la calcula SymPy

# lambdify traduce la expresión simbólica a una función numérica de NumPy.
# Funciona con un número suelto y también con un arreglo completo (al graficar).
f = sp.lambdify((x, D_sym, E_sym), f_expr, "numpy")
df = sp.lambdify((x, D_sym, E_sym), df_expr, "numpy")

display(Math(r"f(x) = " + sp.latex(f_expr, order="none").replace(r"\log", r"\ln")))
display(Math(r"f'(x) = " + sp.latex(df_expr, order="none").replace(r"\log", r"\ln")))

### Ayudas de presentación *(no forma parte del informe exportado)*

Funciones que convierten cada iteración en ecuaciones LaTeX y en gráficas. Se
escriben una sola vez y se reutilizan en los tres métodos.

In [ ]:
def a_latex(expresion, decimales=6):
    """Expresión de SymPy -> texto LaTeX legible.

    decimales=None deja la expresión simbólica tal cual (forma general).
    """
    if decimales is not None:
        expresion = sp.N(expresion, decimales)

    # order="none" conserva el orden natural: primero el término con ln.
    texto = sp.latex(expresion, order="none").replace(r"\log", r"\ln")

    # SymPy escribe "9.0 \ln(x)"; se lee mejor como "9 \ln(x)".
    return re.sub(r"(\d)\.0(?!\d)", r"\1", texto)


def num(valor, decimales=6):
    """Número -> texto para LaTeX, con notación científica si es muy pequeño."""
    valor = float(valor)

    if valor != 0 and abs(valor) < 1e-4:
        base, exponente = f"{valor:.4e}".split("e")
        return rf"{base} \times 10^{{{int(exponente)}}}"

    # Quitamos los ceros sobrantes: 200.000000 se lee mejor como 200.
    return f"{valor:.{decimales}f}".rstrip("0").rstrip(".")


def mostrar_ecuacion_actual(D, E):
    """Ecuación general + ecuación y derivada con los valores actuales de D y E."""
    display(Markdown("**Ecuación general**"))
    display(Math(r"f(x) = " + a_latex(f_expr, decimales=None)))

    display(Markdown("**Valores del caso analizado**"))
    display(Math(rf"D = {D} \qquad E = {E}"))

    valores = {D_sym: D, E_sym: E}
    display(Math(r"f(x) = " + a_latex(f_expr.subs(valores))))
    display(Math(r"f'(x) = " + a_latex(df_expr.subs(valores))))


# Nota: dentro de una f-string, "{{" y "}}" producen las llaves literales "{" y "}"
# que LaTeX necesita para los subíndices.
def mostrar_paso_biseccion(fila, D, E):
    """Desarrollo matemático de una iteración de Bisección."""
    k = int(fila["iteración"])
    a, b, m = fila["a"], fila["b"], fila["m"]
    fa, fm = f(a, D, E), f(m, D, E)

    display(Math(rf"a_{{{k}}} = {num(a)} \qquad b_{{{k}}} = {num(b)}"))
    display(Math(
        rf"m_{{{k}}} = \frac{{a_{{{k}}} + b_{{{k}}}}}{{2}}"
        rf" = \frac{{{num(a)} + {num(b)}}}{{2}} = {num(m)}"
    ))
    display(Math(rf"f(a_{{{k}}}) = {num(fa)} \qquad f(m_{{{k}}}) = {num(fm)}"))

    if fa * fm < 0:
        # La raíz quedó entre a y m: se conserva la mitad izquierda.
        display(Math(rf"f(a_{{{k}}})\,f(m_{{{k}}}) < 0"))
        display(Math(
            rf"[a_{{{k + 1}}}, b_{{{k + 1}}}] = [a_{{{k}}}, m_{{{k}}}]"
            rf" = [{num(a)},\; {num(m)}]"
        ))
    else:
        display(Math(rf"f(a_{{{k}}})\,f(m_{{{k}}}) > 0"))
        display(Math(
            rf"[a_{{{k + 1}}}, b_{{{k + 1}}}] = [m_{{{k}}}, b_{{{k}}}]"
            rf" = [{num(m)},\; {num(b)}]"
        ))

    display(Math(
        rf"b_{{{k}}} - a_{{{k}}} = {num(b - a)} \quad\longrightarrow\quad"
        rf" b_{{{k + 1}}} - a_{{{k + 1}}} = {num((b - a) / 2)}"
    ))


def mostrar_paso_regla_falsa(fila, D, E):
    """Desarrollo matemático de una iteración de Regla Falsa."""
    k = int(fila["iteración"])
    a, b, xr = fila["a"], fila["b"], fila["xr"]
    fa, fb, fr = f(a, D, E), f(b, D, E), f(xr, D, E)

    display(Math(r"x_r = \frac{a\,f(b) - b\,f(a)}{f(b) - f(a)}"))
    display(Math(rf"a_{{{k}}} = {num(a)} \qquad b_{{{k}}} = {num(b)}"))
    display(Math(rf"f(a_{{{k}}}) = {num(fa)} \qquad f(b_{{{k}}}) = {num(fb)}"))
    display(Math(
        rf"x_{{r,{k}}} = \frac{{({num(a)})({num(fb)}) - ({num(b)})({num(fa)})}}"
        rf"{{{num(fb)} - {num(fa)}}} = {num(xr)}"
    ))
    display(Math(rf"f(x_{{r,{k}}}) = {num(fr)}"))

    if fa * fr < 0:
        display(Math(rf"f(a_{{{k}}})\,f(x_{{r,{k}}}) < 0"))
        display(Math(
            rf"[a_{{{k + 1}}}, b_{{{k + 1}}}] = [a_{{{k}}}, x_{{r,{k}}}]"
            rf" = [{num(a)},\; {num(xr)}]"
        ))
    else:
        display(Math(rf"f(a_{{{k}}})\,f(x_{{r,{k}}}) > 0"))
        display(Math(
            rf"[a_{{{k + 1}}}, b_{{{k + 1}}}] = [x_{{r,{k}}}, b_{{{k}}}]"
            rf" = [{num(xr)},\; {num(b)}]"
        ))


def mostrar_paso_newton(fila, D, E):
    """Desarrollo matemático de una iteración de Newton-Raphson."""
    k = int(fila["iteración"])
    xn = fila["xn"]              # valor con el que entra la iteración: x_{k-1}
    xs = fila["xn+1"]            # valor que produce la iteración: x_k
    fx, dfx = fila["f(xn)"], fila["f'(xn)"]

    display(Math(r"x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}"))
    display(Math(
        rf"x_{{{k}}} = x_{{{k - 1}}} - \frac{{f(x_{{{k - 1}}})}}{{f'(x_{{{k - 1}}})}}"
        rf" = {num(xn)} - \frac{{f({num(xn)})}}{{f'({num(xn)})}}"
    ))
    display(Math(
        rf"x_{{{k}}} = {num(xn)} - \frac{{{num(fx)}}}{{{num(dfx)}}} = {num(xs)}"
    ))
    display(Math(rf"f(x_{{{k}}}) = {num(f(xs, D, E))}"))
    display(Math(rf"|x_{{{k}}} - x_{{{k - 1}}}| = {num(abs(xs - xn))}"))

In [ ]:
def dibujar_biseccion(fila, D, E):
    """Gráfica de una iteración de Bisección (intervalo y punto medio)."""
    k = int(fila["iteración"])
    a, b, m = fila["a"], fila["b"], fila["m"]

    margen = max(2, (b - a) * 0.35)
    xs = np.linspace(max(0.1, a - margen), b + margen, 800)

    plt.figure()
    plt.plot(xs, f(xs, D, E), label="f(x)")
    plt.axhline(0, linewidth=1, color="black")
    plt.axvspan(a, b, alpha=0.12, label=f"[a, b], ancho = {b - a:.4f}")
    plt.axvline(m, linestyle=":", linewidth=2, color="tab:red", label=f"m = {m:.6f}")
    plt.scatter([a, m, b], [f(a, D, E), f(m, D, E), f(b, D, E)], s=70, zorder=5)

    for etiqueta, punto in (("a", a), ("m", m), ("b", b)):
        plt.annotate(etiqueta, (punto, f(punto, D, E)),
                     textcoords="offset points", xytext=(0, 10))

    plt.title(f"Bisección — iteración {k}   (D={D}, E={E})")
    plt.xlabel("x (segundos)")
    plt.ylabel("f(x)")
    plt.legend()
    plt.show()


def dibujar_regla_falsa(fila, D, E):
    """Gráfica de una iteración de Regla Falsa (recta secante y su corte)."""
    k = int(fila["iteración"])
    a, b, xr = fila["a"], fila["b"], fila["xr"]
    fa, fb, fr = f(a, D, E), f(b, D, E), f(xr, D, E)

    margen = max(2, (b - a) * 0.25)
    xs = np.linspace(max(0.1, a - margen), b + margen, 900)

    # Recta secante que une (a, f(a)) con (b, f(b)).
    sec_x = np.linspace(a, b, 100)
    sec_y = fa + (fb - fa) * (sec_x - a) / (b - a)

    plt.figure()
    plt.plot(xs, f(xs, D, E), label="f(x)")
    plt.plot(sec_x, sec_y, linestyle="--", color="tab:orange", label="Recta secante A-B")
    plt.axhline(0, linewidth=1, color="black")
    plt.axvspan(a, b, alpha=0.08, label="Intervalo actual")

    plt.scatter([a, b], [fa, fb], s=80, color="tab:blue", zorder=5)
    plt.annotate(f"A = ({a:.3f}, {fa:.3f})", (a, fa),
                 textcoords="offset points", xytext=(5, 10))
    plt.annotate(f"B = ({b:.3f}, {fb:.3f})", (b, fb),
                 textcoords="offset points", xytext=(-40, -18))

    plt.scatter([xr], [0], s=120, marker="x", color="tab:red", zorder=6,
                label=f"xr = {xr:.6f} (corte con el eje x)")
    plt.scatter([xr], [fr], s=70, color="tab:green", zorder=6, label="(xr, f(xr))")
    plt.plot([xr, xr], [0, fr], linestyle=":", color="tab:green")

    plt.title(f"Regla Falsa — iteración {k}   (D={D}, E={E})")
    plt.xlabel("x (segundos)")
    plt.ylabel("f(x)")
    plt.legend()
    plt.show()


def dibujar_newton(fila, D, E):
    """Gráfica de una iteración de Newton-Raphson (recta tangente y su corte)."""
    k = int(fila["iteración"])
    xn, siguiente = fila["xn"], fila["xn+1"]
    fx, pendiente = fila["f(xn)"], fila["f'(xn)"]

    izquierda = max(0.1, min(xn, siguiente) - 12)
    derecha = max(xn, siguiente) + 12

    xs = np.linspace(izquierda, derecha, 900)
    tangente = fx + pendiente * (xs - xn)   # recta tangente en xn

    plt.figure()
    plt.plot(xs, f(xs, D, E), label="f(x)")
    plt.plot(xs, tangente, linestyle="--", color="tab:orange",
             label=f"Tangente en xn (pendiente = {pendiente:.6f})")
    plt.axhline(0, linewidth=1, color="black")

    plt.scatter([xn], [fx], s=90, color="tab:blue", zorder=5,
                label=f"(xn, f(xn)) = ({xn:.4f}, {fx:.4f})")
    plt.scatter([siguiente], [0], s=130, marker="x", color="tab:red", zorder=6,
                label=f"xn+1 = {siguiente:.6f}")
    plt.scatter([siguiente], [f(siguiente, D, E)], s=70, color="tab:green", zorder=6,
                label="(xn+1, f(xn+1))")

    # Guías verticales para ver el salto de xn hasta xn+1.
    plt.plot([xn, xn], [0, fx], linestyle=":", color="tab:blue")
    plt.plot([siguiente, siguiente], [0, f(siguiente, D, E)],
             linestyle=":", color="tab:green")

    plt.title(f"Newton-Raphson — iteración {k}   (D={D}, E={E})")
    plt.xlabel("x (segundos)")
    plt.ylabel("f(x)")
    plt.legend()
    plt.show()


def informe_iteraciones(tabla, dibujar, mostrar_paso, D, E, detalladas=(0, -1)):
    """Documenta un método: tabla completa + desarrollo de algunas iteraciones.

    detalladas: posiciones de la tabla que se explican paso a paso
    (por defecto la primera y la última).
    """
    display(Markdown("**Tabla de iteraciones**"))
    display(tabla.round(6))

    ya_mostradas = []

    for posicion in detalladas:
        fila = tabla.iloc[posicion]
        numero = int(fila["iteración"])

        # Si el método necesitó una sola iteración, primera y última coinciden.
        if numero in ya_mostradas:
            continue
        ya_mostradas.append(numero)

        display(Markdown(f"**Iteración {numero} de {len(tabla)} — desarrollo**"))
        dibujar(fila, D, E)
        mostrar_paso(fila, D, E)

# 2. Configuración del caso analizado

El documento se genera a partir de los siguientes parámetros:

| Parámetro | Significado |
| --- | --- |
| $D$, $E$ | constantes de la ecuación |
| $[a, b]$ | intervalo inicial de Bisección y Regla Falsa |
| $x_0$ | aproximación inicial de Newton-Raphson |
| tolerancia | criterio de parada $\lvert f(x_k)\rvert < \text{tol}$ |

Los valores del caso base son $D = 9$, $E = 6$, $[a, b] = [200, 250]$,
$x_0 = 225$ y tolerancia $0.001$. Están definidos en una única celda de
parámetros: al cambiarlos y volver a ejecutar el documento, **todo el informe se
recalcula** (ecuaciones, tablas, gráficas y conclusiones numéricas).

### 2.1 Parámetros y función resultante

In [ ]:
# --- Parámetros del caso analizado --------------------------------------
# Cambia estos valores y vuelve a ejecutar el documento completo
# (Kernel -> Restart & Run All) para regenerar el informe con otro caso.
D = 9          # constante que acompaña a ln(x)
E = 6          # constante que acompaña a la exponencial
a = 200.0      # extremo izquierdo del intervalo
b = 250.0      # extremo derecho del intervalo
x0 = 225.0     # aproximación inicial de Newton-Raphson
tol = 0.001    # criterio de parada: |f(x)| < tol

mostrar_ecuacion_actual(D, E)

display(Math(
    rf"[a, b] = [{num(a, 4)},\; {num(b, 4)}] \qquad "
    rf"x_0 = {num(x0, 4)} \qquad"
    rf"\text{{tol}} = {tol}"
))

### Controles interactivos *(no forma parte del informe exportado)*

Estos controles sirven para **explorar** sin modificar el informe: al moverlos se
actualizan la ecuación mostrada aquí y los exploradores iteración por iteración de
cada método. Si el intervalo deja de encerrar una raíz, **Autoajustar intervalo**
busca un cambio de signo y reubica $[a, b]$ y $x_0$.

Para que el **informe** use otros valores, edita la celda de parámetros de la
sección 2.1 y vuelve a ejecutar el documento
(*Runtime → Run all* / *Kernel → Restart & Run All*).

In [ ]:
D_widget = widgets.IntSlider(value=D, min=1, max=9, step=1,
                             description="D:", continuous_update=False)
E_widget = widgets.IntSlider(value=E, min=1, max=9, step=1,
                             description="E:", continuous_update=False)

a_widget = widgets.FloatText(value=a, description="a:")
b_widget = widgets.FloatText(value=b, description="b:")
x0_widget = widgets.FloatText(value=x0, description="x₀:")

tol_widget = widgets.SelectionSlider(
    options=[("0.1", 1e-1), ("0.01", 1e-2), ("0.001", 1e-3),
             ("0.0001", 1e-4), ("0.00001", 1e-5)],
    value=tol,
    description="Tolerancia:",
    continuous_update=False,
    style={"description_width": "initial"},
)

auto_button = widgets.Button(description="Autoajustar intervalo", button_style="info")
auto_output = widgets.Output()
ecuacion_output = widgets.Output()


def config_actual():
    """Valores actuales de los controles, reunidos en un diccionario."""
    return {
        "D": D_widget.value,
        "E": E_widget.value,
        "a": a_widget.value,
        "b": b_widget.value,
        "x0": x0_widget.value,
        "tol": tol_widget.value,
    }


def buscar_intervalo(D, E, xmin=1.0, xmax=1000.0, muestras=10000):
    """Busca un cambio de signo de f entre xmin y xmax."""
    xs = np.linspace(xmin, xmax, muestras)
    ys = f(xs, D, E)          # f está vectorizada gracias a lambdify

    cambios = [(xs[i], xs[i + 1]) for i in range(len(xs) - 1)
               if ys[i] == 0 or ys[i] * ys[i + 1] < 0]

    if not cambios:
        return None

    # Escogemos la raíz positiva más grande: el punto de retorno del escenario.
    return cambios[-1]


def autoajustar(_):
    auto_output.clear_output()

    with auto_output:
        intervalo = buscar_intervalo(D_widget.value, E_widget.value)

        if intervalo is None:
            print("No encontré un cambio de signo entre 1 y 1000 segundos.")
            return

        a, b = intervalo
        centro = (a + b) / 2

        a_widget.value = max(0.1, centro - 10)
        b_widget.value = centro + 10
        x0_widget.value = centro

        print(f"Intervalo ajustado a [{a_widget.value:.4f}, {b_widget.value:.4f}]")


def refrescar_ecuacion(_=None):
    """Se ejecuta cada vez que cambian D o E."""
    ecuacion_output.clear_output()

    with ecuacion_output:
        mostrar_ecuacion_actual(D_widget.value, E_widget.value)


auto_button.on_click(autoajustar)
D_widget.observe(refrescar_ecuacion, names="value")
E_widget.observe(refrescar_ecuacion, names="value")

display(
    widgets.HBox([D_widget, E_widget]),
    widgets.HBox([a_widget, b_widget, x0_widget]),
    tol_widget,
    auto_button,
    auto_output,
    ecuacion_output,
)

refrescar_ecuacion()

### 2.2 Localización gráfica de la raíz

Antes de aplicar cualquier método se verifica que el intervalo $[a, b]$ encierre
una raíz. Por el teorema del valor intermedio basta con que la función cambie de
signo en los extremos:

$$
f(a)\,f(b) < 0 \;\Longrightarrow\; \exists\, r \in (a, b) \text{ tal que } f(r) = 0
$$

In [ ]:
margen = max(10, (b - a) * 0.25)
xs = np.linspace(max(0.1, a - margen), b + margen, 1000)

fa, fb = f(a, D, E), f(b, D, E)

plt.figure()
plt.plot(xs, f(xs, D, E), label=f"f(x)   D={D}, E={E}")
plt.axhline(0, linewidth=1, color="black")
plt.axvspan(a, b, alpha=0.10, label="[a, b]")
plt.scatter([a, b], [fa, fb], s=70, zorder=5)
plt.annotate("a", (a, fa), textcoords="offset points", xytext=(0, 10))
plt.annotate("b", (b, fb), textcoords="offset points", xytext=(0, 10))
plt.title("Función de balance de energía y localización de la raíz")
plt.xlabel("x (segundos)")
plt.ylabel("f(x)")
plt.legend()
plt.show()

display(Math(rf"f(a) = {num(fa)} \qquad f(b) = {num(fb)}"))

if fa * fb < 0:
    display(Math(r"f(a)\,f(b) < 0 \;\Rightarrow\; \text{existe raíz en } [a, b]"))
else:
    display(Math(r"f(a)\,f(b) > 0 \;\Rightarrow\; \text{no se garantiza raíz en } [a, b]"))

# 3. Bisección

Partiendo de un intervalo $[a_k, b_k]$ con cambio de signo, se calcula su punto medio

$$
m_k = \frac{a_k + b_k}{2}
$$

y se conserva la mitad donde persiste el cambio de signo:

$$
\begin{aligned}
f(a_k)\,f(m_k) &< 0 &&\Longrightarrow&& [a_{k+1}, b_{k+1}] = [a_k, m_k] \\[4pt]
f(a_k)\,f(m_k) &> 0 &&\Longrightarrow&& [a_{k+1}, b_{k+1}] = [m_k, b_k]
\end{aligned}
$$

El ancho del intervalo se reduce exactamente a la mitad en cada paso, de modo que

$$
b_k - a_k = \frac{b_0 - a_0}{2^{\,k}}
\qquad\Longrightarrow\qquad
|m_k - r| \le \frac{b_0 - a_0}{2^{\,k+1}}
$$

El método es lento pero **siempre converge** si el intervalo inicial encierra una raíz.

In [ ]:
def biseccion(D, E, a, b, tolerancia=0.001, max_iter=100):
    """Método de Bisección. Devuelve una tabla con todas las iteraciones."""
    fa = f(a, D, E)
    fb = f(b, D, E)

    if fa * fb >= 0:
        raise ValueError("Bisección necesita que f(a) y f(b) tengan signos diferentes.")

    filas = []
    anterior = None

    for i in range(1, max_iter + 1):
        m = (a + b) / 2
        fm = f(m, D, E)

        filas.append({
            "iteración": i,
            "a": a,
            "b": b,
            "m": m,
            "f(m)": fm,
            "|f(m)|": abs(fm),
            "error_x": None if anterior is None else abs(m - anterior),
            "ancho": b - a,
        })

        if abs(fm) < tolerancia:
            break

        if fa * fm < 0:
            b = m
            fb = fm
        else:
            a = m
            fa = fm

        anterior = m

    return pd.DataFrame(filas)

### 3.1 Resultados

In [ ]:
tabla_bis = biseccion(D, E, a, b, tol)

informe_iteraciones(tabla_bis, dibujar_biseccion, mostrar_paso_biseccion,
                    D, E)

display(Math(
    rf"\text{{Raíz aproximada}} \approx {num(tabla_bis.iloc[-1]['m'])}"
    rf" \quad\text{{en {len(tabla_bis)} iteraciones}}"
))

### Explorador iteración por iteración *(no forma parte del informe exportado)*

In [ ]:
def explorar_biseccion(iteracion):
    """Muestra la iteración pedida usando los valores actuales de los controles."""
    actual = config_actual()

    try:
        tabla = biseccion(actual["D"], actual["E"], actual["a"], actual["b"],
                          actual["tol"])
    except ValueError as error:
        print(error)
        return

    # Si el método terminó antes, mostramos la última iteración disponible.
    fila = tabla.iloc[min(iteracion, len(tabla)) - 1]

    dibujar_biseccion(fila, actual["D"], actual["E"])
    mostrar_paso_biseccion(fila, actual["D"], actual["E"])


widgets.interact(
    explorar_biseccion,
    iteracion=widgets.IntSlider(value=1, min=1, max=40, step=1,
                                description="Iteración:", continuous_update=False,
                                style={"description_width": "initial"},
                                layout=widgets.Layout(width="70%")),
);

# 4. Regla Falsa

En lugar del punto medio se utiliza el corte con el eje $x$ de la **recta secante**
que une los puntos $(a, f(a))$ y $(b, f(b))$:

$$
x_r = \frac{a\,f(b) - b\,f(a)}{f(b) - f(a)}
$$

La regla de selección del nuevo intervalo es la misma de Bisección, sustituyendo
$m_k$ por $x_{r,k}$:

$$
\begin{aligned}
f(a_k)\,f(x_{r,k}) &< 0 &&\Longrightarrow&& [a_{k+1}, b_{k+1}] = [a_k, x_{r,k}] \\[4pt]
f(a_k)\,f(x_{r,k}) &> 0 &&\Longrightarrow&& [a_{k+1}, b_{k+1}] = [x_{r,k}, b_k]
\end{aligned}
$$

Al aprovechar el **valor** de la función y no solo su signo, suele necesitar menos
iteraciones que Bisección; a cambio, uno de los extremos puede quedar fijo durante
varias iteraciones.

In [ ]:
def regla_falsa(D, E, a, b, tolerancia=0.001, max_iter=100):
    """Método de Regla Falsa. Devuelve una tabla con todas las iteraciones."""
    fa = f(a, D, E)
    fb = f(b, D, E)

    if fa * fb >= 0:
        raise ValueError("Regla Falsa necesita que f(a) y f(b) tengan signos diferentes.")

    filas = []
    anterior = None

    for i in range(1, max_iter + 1):
        xr = (a * fb - b * fa) / (fb - fa)
        fr = f(xr, D, E)

        filas.append({
            "iteración": i,
            "a": a,
            "b": b,
            "xr": xr,
            "f(xr)": fr,
            "|f(xr)|": abs(fr),
            "error_x": None if anterior is None else abs(xr - anterior),
        })

        if abs(fr) < tolerancia:
            break

        if fa * fr < 0:
            b = xr
            fb = fr
        else:
            a = xr
            fa = fr

        anterior = xr

    return pd.DataFrame(filas)

### 4.1 Resultados

In [ ]:
tabla_rf = regla_falsa(D, E, a, b, tol)

informe_iteraciones(tabla_rf, dibujar_regla_falsa, mostrar_paso_regla_falsa,
                    D, E)

display(Math(
    rf"\text{{Raíz aproximada}} \approx {num(tabla_rf.iloc[-1]['xr'])}"
    rf" \quad\text{{en {len(tabla_rf)} iteraciones}}"
))

### Explorador iteración por iteración *(no forma parte del informe exportado)*

In [ ]:
def explorar_regla_falsa(iteracion):
    actual = config_actual()

    try:
        tabla = regla_falsa(actual["D"], actual["E"], actual["a"], actual["b"],
                            actual["tol"])
    except ValueError as error:
        print(error)
        return

    fila = tabla.iloc[min(iteracion, len(tabla)) - 1]

    dibujar_regla_falsa(fila, actual["D"], actual["E"])
    mostrar_paso_regla_falsa(fila, actual["D"], actual["E"])


widgets.interact(
    explorar_regla_falsa,
    iteracion=widgets.IntSlider(value=1, min=1, max=40, step=1,
                                description="Iteración:", continuous_update=False,
                                style={"description_width": "initial"},
                                layout=widgets.Layout(width="70%")),
);

# 5. Newton-Raphson

A partir de una aproximación $x_n$ se traza la **recta tangente** a la curva:

$$
y = f(x_n) + f'(x_n)\,(x - x_n)
$$

Igualando $y = 0$ se obtiene el punto donde la tangente corta el eje $x$, que es la
siguiente aproximación:

$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}, \qquad f'(x_n) \neq 0
$$

Con la derivada obtenida simbólicamente:

$$
f'(x) = \frac{D}{x} - 0.002\,E\,e^{0.02x}
$$

No requiere intervalo, pero sí una aproximación inicial adecuada. Cuando converge,
lo hace de forma **cuadrática**: aproximadamente duplica las cifras correctas en
cada iteración.

In [ ]:
def newton_raphson(D, E, x0, tolerancia=0.001, max_iter=100):
    """Método de Newton-Raphson. Devuelve una tabla con todas las iteraciones."""
    if x0 <= 0:
        raise ValueError("x₀ debe ser positivo.")

    filas = []
    xn = x0

    for i in range(1, max_iter + 1):
        fx = f(xn, D, E)
        dfx = df(xn, D, E)

        if abs(dfx) < 1e-12:
            raise ValueError("La derivada es demasiado cercana a cero.")

        x_nuevo = xn - fx / dfx

        if x_nuevo <= 0:
            raise ValueError("Newton salió del dominio x > 0. Prueba otro x₀.")

        filas.append({
            "iteración": i,
            "xn": xn,
            "f(xn)": fx,
            "f'(xn)": dfx,
            "xn+1": x_nuevo,
            "|f(xn+1)|": abs(f(x_nuevo, D, E)),
            "error_x": abs(x_nuevo - xn),
        })

        if abs(f(x_nuevo, D, E)) < tolerancia:
            break

        xn = x_nuevo

    return pd.DataFrame(filas)

### 5.1 Resultados

In [ ]:
tabla_newton = newton_raphson(D, E, x0, tol)

informe_iteraciones(tabla_newton, dibujar_newton, mostrar_paso_newton,
                    D, E)

display(Math(
    rf"\text{{Raíz aproximada}} \approx {num(tabla_newton.iloc[-1]['xn+1'])}"
    rf" \quad\text{{en {len(tabla_newton)} iteraciones}}"
))

### Explorador iteración por iteración *(no forma parte del informe exportado)*

In [ ]:
def explorar_newton(iteracion):
    actual = config_actual()

    try:
        tabla = newton_raphson(actual["D"], actual["E"], actual["x0"], actual["tol"])
    except ValueError as error:
        print(error)
        return

    fila = tabla.iloc[min(iteracion, len(tabla)) - 1]

    dibujar_newton(fila, actual["D"], actual["E"])
    mostrar_paso_newton(fila, actual["D"], actual["E"])


widgets.interact(
    explorar_newton,
    iteracion=widgets.IntSlider(value=1, min=1, max=40, step=1,
                                description="Iteración:", continuous_update=False,
                                style={"description_width": "initial"},
                                layout=widgets.Layout(width="70%")),
);

# 6. Comparación de los tres métodos

Los tres métodos se ejecutan sobre la misma función, con la misma tolerancia y la
misma configuración. Las columnas de la tabla son:

| Columna | Significado |
| --- | --- |
| Iteraciones | pasos hasta cumplir $\lvert f(x_k)\rvert < \text{tol}$ |
| Raíz aproximada | último valor calculado |
| Residuo $\lvert f(x_k)\rvert$ | qué tan cerca de cero quedó la función |
| Cambio final $\lvert x_k - x_{k-1}\rvert$ | desplazamiento de la aproximación en el último paso |

La última columna se denomina **cambio final** y no "error" porque la raíz exacta es
desconocida: se trata de una *estimación* del error y es la única métrica directamente
comparable entre los tres métodos. Bisección dispone además de la cota teórica
$\frac{b_0 - a_0}{2^{\,k}}$.

In [ ]:
def cambio_final(tabla):
    """Último |x_k - x_{k-1}| de una tabla; NaN si solo hubo una iteración."""
    valores = tabla["error_x"].dropna()
    return float(valores.iloc[-1]) if len(valores) else float("nan")


comparacion = pd.DataFrame([
    {
        "Método": "Bisección",
        "Iteraciones": len(tabla_bis),
        "Raíz aproximada": tabla_bis.iloc[-1]["m"],
        "Residuo |f(x)|": abs(tabla_bis.iloc[-1]["f(m)"]),
        "Cambio final |Δx|": cambio_final(tabla_bis),
    },
    {
        "Método": "Regla Falsa",
        "Iteraciones": len(tabla_rf),
        "Raíz aproximada": tabla_rf.iloc[-1]["xr"],
        "Residuo |f(x)|": abs(tabla_rf.iloc[-1]["f(xr)"]),
        "Cambio final |Δx|": cambio_final(tabla_rf),
    },
    {
        "Método": "Newton-Raphson",
        "Iteraciones": len(tabla_newton),
        "Raíz aproximada": tabla_newton.iloc[-1]["xn+1"],
        "Residuo |f(x)|": tabla_newton.iloc[-1]["|f(xn+1)|"],
        "Cambio final |Δx|": cambio_final(tabla_newton),
    },
])

display(comparacion.round(8))

### 6.1 Iteraciones requeridas

In [ ]:
plt.figure(figsize=(9, 4.5))
barras = plt.bar(comparacion["Método"], comparacion["Iteraciones"],
                 color=["tab:blue", "tab:orange", "tab:green"])
plt.bar_label(barras)
plt.title(f"Iteraciones para alcanzar |f(x)| < {tol}")
plt.ylabel("Iteraciones")
plt.show()

### 6.2 Aproximación a la raíz

In [ ]:
plt.figure()
plt.plot(tabla_bis["iteración"], tabla_bis["m"], marker="o", label="Bisección")
plt.plot(tabla_rf["iteración"], tabla_rf["xr"], marker="s", label="Regla Falsa")
plt.plot(tabla_newton["iteración"], tabla_newton["xn+1"], marker="^",
         label="Newton-Raphson")
plt.axhline(tabla_newton.iloc[-1]["xn+1"], linestyle="--", color="gray",
            label="raíz de referencia")
plt.title("Aproximación a la raíz por iteración")
plt.xlabel("Iteración")
plt.ylabel("Aproximación")
plt.legend()
plt.show()

### 6.3 Residuo en escala logarítmica

La escala logarítmica convierte "dividir entre 10" en un salto constante: cuanto más
inclinada la curva, más rápido gana precisión el método.

In [ ]:
plt.figure()
plt.semilogy(tabla_bis["iteración"], tabla_bis["|f(m)|"], marker="o", label="Bisección")
plt.semilogy(tabla_rf["iteración"], tabla_rf["|f(xr)|"], marker="s", label="Regla Falsa")
plt.semilogy(tabla_newton["iteración"], tabla_newton["|f(xn+1)|"], marker="^",
             label="Newton-Raphson")
plt.axhline(tol, linestyle="--", color="gray", label=f"tolerancia = {tol}")
plt.title("Residuo |f(x)| por iteración (escala logarítmica)")
plt.xlabel("Iteración")
plt.ylabel("|f(x)|")
plt.grid(alpha=0.25, which="both")
plt.legend()
plt.show()

# 7. Convergencia

Una sucesión de aproximaciones $x_1, x_2, x_3, \dots$ **converge** a la raíz $r$
cuando se acerca progresivamente a ella:

$$
|x_k - r| \longrightarrow 0 \quad \text{cuando } k \to \infty
$$

En las gráficas anteriores esto se observa de dos maneras: en la sección 6.2 las tres
curvas se aplanan hacia un mismo valor, y en la 6.3 el residuo desciende hasta cruzar
la tolerancia.

Para Bisección la convergencia admite una garantía inmediata: el intervalo se reduce
a la mitad en cada iteración, por lo que el error está acotado por
$\frac{b_0 - a_0}{2^{\,k+1}}$ independientemente de la forma de la función. La
siguiente gráfica compara el ancho real del intervalo con esa cota teórica.

In [ ]:
ancho_inicial = tabla_bis.iloc[0]["ancho"]
iteraciones = tabla_bis["iteración"]
teorico = ancho_inicial / 2 ** (iteraciones - 1)

plt.figure()
plt.semilogy(iteraciones, tabla_bis["ancho"], marker="o", label="ancho real b − a")
plt.semilogy(iteraciones, teorico, linestyle="--", label="(b₀ − a₀) / 2^(k−1)")
plt.title("Bisección: reducción del intervalo a la mitad en cada iteración")
plt.xlabel("Iteración")
plt.ylabel("Ancho del intervalo")
plt.grid(alpha=0.25, which="both")
plt.legend()
plt.show()

display(Math(
    rf"b_1 - a_1 = {num(ancho_inicial)} \qquad "
    rf"b_{{{len(tabla_bis)}}} - a_{{{len(tabla_bis)}}}"
    rf" = {num(tabla_bis.iloc[-1]['ancho'])}"
))
display(Markdown(
    f"En {len(tabla_bis)} iteraciones el intervalo se redujo "
    f"{ancho_inicial / tabla_bis.iloc[-1]['ancho']:.1f} veces."
))

# 8. Conclusiones

| Método | Información que utiliza | Garantía de convergencia | Velocidad |
| --- | --- | --- | --- |
| Bisección | solo el **signo** de $f$ en los extremos | asegurada si $f(a)f(b) < 0$ | lenta (lineal, factor $1/2$) |
| Regla Falsa | signo **y valor** de $f$ en los extremos | asegurada si $f(a)f(b) < 0$ | intermedia; un extremo puede estancarse |
| Newton-Raphson | valor de $f$ y de $f'$ | no asegurada; depende de $x_0$ | cuadrática cerca de la raíz |

1. Los tres métodos convergen a la **misma raíz** de
   $f(x) = D\ln(x) - \frac{E}{10}e^{0.02x}$, pero utilizan información distinta:
   el punto medio del intervalo, la recta secante o la recta tangente.
2. Cuanta más información se usa por iteración, **menos iteraciones** se requieren,
   a costa de mayores exigencias: Newton-Raphson necesita la derivada y una
   aproximación inicial adecuada.
3. Bisección es el único método con una cota de error conocida de antemano, lo que lo
   hace preferible cuando la robustez importa más que la velocidad.
4. No existe un método universalmente superior: la elección depende de la estabilidad
   exigida, de la información disponible y del costo de cada evaluación de la función.

# 9. Anexo — código de los algoritmos

Implementación de los tres métodos tal como fueron ejecutados en este documento.

In [ ]:
# inspect.getsource lee el código fuente real de cada función, así el anexo
# nunca queda desactualizado respecto a lo que se ejecutó.
for funcion in (biseccion, regla_falsa, newton_raphson):
    try:
        display(Markdown("```python\n" + inspect.getsource(funcion) + "```"))
    except OSError:
        # Algunos entornos no guardan el código de las celdas; no es motivo
        # para interrumpir la ejecución del documento.
        display(Markdown(f"*(no se pudo leer el código de `{funcion.__name__}`)*"))